<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">처음부터 만드는 대형 언어 모델</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 4장: 텍스트 생성을 위한 GPT 모델 처음부터 구현하기

In [ ]:
from importlib.metadata import version

print("matplotlib version:", version("matplotlib"))
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

- 이 장에서는 GPT와 같은 LLM 아키텍처를 구현합니다. 다음 장에서는 이 LLM을 훈련하는 데 중점을 둘 것입니다.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/01.webp" width="500px">

## 4.1 LLM 아키텍처 코딩하기

- 1장에서는 단어를 순차적으로 생성하며 원본 트랜스포머 아키텍처의 디코더 부분을 기반으로 하는 GPT와 Llama와 같은 모델들을 논의했습니다
- 따라서 이러한 LLM들은 종종 "디코더형" LLM이라고 불립니다
- 기존 딥러닝 모델과 비교했을 때, LLM은 더 큽니다. 주로 코드의 양이 아닌 방대한 수의 파라미터 때문입니다
- LLM 아키텍처에서 많은 요소들이 반복된다는 것을 알 수 있습니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/02.webp" width="400px">

- 이전 장들에서는 토큰 입력과 출력에 작은 임베딩 차원을 사용했습니다. 설명을 위해 한 페이지에 맞도록 하기 위함이었습니다
- 이 장에서는 작은 GPT-2 모델과 유사한 임베딩과 모델 크기를 고려합니다
- 구체적으로 Radford et al.의 [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)에서 설명된 가장 작은 GPT-2 모델(1억 2400만 파라미터)의 아키텍처를 코딩합니다 (초기 보고서에서는 1억 1700만 파라미터로 나와 있지만, 이는 나중에 모델 가중치 저장소에서 수정되었습니다)
- 6장에서는 사전 훈련된 가중치를 우리의 구현에 로드하는 방법을 보여드릴 것입니다. 이는 3억 4500만, 7억 6200만, 15억 4200만 파라미터 모델 크기와 호환될 것입니다

- 1억 2400만 파라미터 GPT-2 모델의 구성 세부사항은 다음과 같습니다:

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # 어휘 크기 (Vocabulary size)
    "context_length": 1024, # 컨텍스트 길이 (Context length)
    "emb_dim": 768,         # 임베딩 차원 (Embedding dimension)
    "n_heads": 12,          # 어텐션 헤드 수 (Number of attention heads)
    "n_layers": 12,         # 레이어 수 (Number of layers)
    "drop_rate": 0.1,       # 드롭아웃 비율 (Dropout rate)
    "qkv_bias": False       # Query-Key-Value 편향 (Query-Key-Value bias)
}

- 나중에 긴 코드 줄을 피하기 위해 짧은 변수명을 사용합니다
- `"vocab_size"`는 2장에서 다룬 BPE 토크나이저가 지원하는 50,257개 단어의 어휘 크기를 나타냅니다
- `"context_length"`는 2장에서 다룬 위치 임베딩으로 가능한 모델의 최대 입력 토큰 수를 나타냅니다
- `"emb_dim"`는 토큰 입력의 임베딩 크기로, 각 입력 토큰을 768차원 벡터로 변환합니다
- `"n_heads"`는 3장에서 구현한 멀티헤드 어텐션 메커니즘의 어텐션 헤드 수입니다
- `"n_layers"`는 향후 섹션에서 구현할 모델 내의 트랜스포머 블록 수입니다
- `"drop_rate"`는 3장에서 논의한 드롭아웃 메커니즘의 강도입니다. 0.1은 과적합을 완화하기 위해 훈련 중에 은닉 유닛의 10%를 삭제한다는 의미입니다
- `"qkv_bias"`는 (3장의) 멀티헤드 어텐션 메커니즘의 `Linear` 레이어가 쿼리(Q), 키(K), 값(V) 텐서를 계산할 때 편향 벡터를 포함할지를 결정합니다. 우리는 이 옵션을 비활성화하는데, 이는 현대 LLM의 표준 관행입니다. 하지만 5장에서 OpenAI의 사전 훈련된 GPT-2 가중치를 우리의 재구현에 로드할 때 이를 다시 검토할 것입니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/03.webp" width="500px">

In [ ]:
import torch
import torch.nn as nn


class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        # TransformerBlock에 대한 플레이스홀더 사용
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        # LayerNorm에 대한 플레이스홀더 사용
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # 간단한 플레이스홀더

    def forward(self, x):
        # 이 블록은 아무것도 하지 않고 단지 입력을 반환합니다.
        return x


class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        # 여기의 파라미터들은 단지 LayerNorm 인터페이스를 모방하기 위한 것입니다.

    def forward(self, x):
        # 이 레이어는 아무것도 하지 않고 단지 입력을 반환합니다.
        return x

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/04.webp?123" width="500px">

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

In [ ]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)

logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

---

**참고**

- Windows나 Linux에서 이 코드를 실행하는 경우, 위의 결과값이 다음과 같이 나타날 수 있습니다:
    
```
Output shape: torch.Size([2, 4, 50257])
tensor([[[-0.9289,  0.2748, -0.7557,  ..., -1.6070,  0.2702, -0.5888],
         [-0.4476,  0.1726,  0.5354,  ..., -0.3932,  1.5285,  0.8557],
         [ 0.5680,  1.6053, -0.2155,  ...,  1.1624,  0.1380,  0.7425],
         [ 0.0447,  2.4787, -0.8843,  ...,  1.3219, -0.0864, -0.5856]],

        [[-1.5474, -0.0542, -1.0571,  ..., -1.8061, -0.4494, -0.6747],
         [-0.8422,  0.8243, -0.1098,  ..., -0.1434,  0.2079,  1.2046],
         [ 0.1355,  1.1858, -0.1453,  ...,  0.0869, -0.1590,  0.1552],
         [ 0.1666, -0.8138,  0.2307,  ...,  2.5035, -0.3055, -0.3083]]],
       grad_fn=<UnsafeViewBackward0>)
```

- 이는 단지 랜덤한 숫자들이므로 걱정할 필요가 없으며, 이 장의 나머지 부분을 문제없이 진행할 수 있습니다
- 이러한 불일치의 한 가지 가능한 이유는 [PyTorch 이슈 트래커에서 논의된 바와 같이](https://github.com/pytorch/pytorch/issues/121595) PyTorch가 컴파일된 방식에 따라 운영체제별로 `nn.Dropout`의 동작이 다르기 때문입니다

---

## 4.2 레이어 정규화로 활성화 정규화하기

- LayerNorm이라고도 알려진 레이어 정규화([Ba et al. 2016](https://arxiv.org/abs/1607.06450))는 신경망 레이어의 활성화를 평균 0 주변으로 중심화하고 분산을 1로 정규화합니다
- 이는 훈련을 안정화하고 효과적인 가중치로의 더 빠른 수렴을 가능하게 합니다
- 레이어 정규화는 나중에 구현할 트랜스포머 블록 내에서 멀티헤드 어텐션 모듈의 전후에 적용되며, 최종 출력 레이어 전에도 적용됩니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/05.webp" width="400px">

- 작은 입력 샘플을 간단한 신경망 레이어를 통과시켜 레이어 정규화가 어떻게 작동하는지 살펴보겠습니다:

In [ ]:
torch.manual_seed(123)

# 각각 5개 차원(특성)을 가진 2개의 훈련 예제를 생성합니다
batch_example = torch.randn(2, 5) 

layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)

- 위의 2개 입력 각각에 대해 평균과 분산을 계산해보겠습니다:

In [ ]:
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)

print("Mean:\n", mean)
print("Variance:\n", var)

- 정규화는 두 입력(행) 각각에 독립적으로 적용됩니다. dim=-1을 사용하면 행 차원 대신 마지막 차원(이 경우 특성 차원)에 대해 계산이 적용됩니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/06.webp" width="400px">

- 평균을 빼고 분산의 제곱근(표준편차)으로 나누면 입력이 열(특성) 차원에서 평균 0과 분산 1을 갖도록 중심화됩니다:

In [ ]:
out_norm = (out - mean) / torch.sqrt(var)
print("Normalized layer outputs:\n", out_norm)

mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)
print("Mean:\n", mean)
print("Variance:\n", var)

- 각 입력은 0을 중심으로 하고 단위 분산 1을 갖습니다. 가독성을 높이기 위해 PyTorch의 과학적 표기법을 비활성화할 수 있습니다:

In [ ]:
torch.set_printoptions(sci_mode=False)
print("Mean:\n", mean)
print("Variance:\n", var)

- 위에서는 각 입력의 특성을 정규화했습니다
- 이제 같은 아이디어를 사용하여 `LayerNorm` 클래스를 구현할 수 있습니다:

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

**스케일과 시프트**

- 평균을 빼고 분산으로 나누어 정규화를 수행하는 것 외에도, 두 개의 훈련 가능한 파라미터인 `scale`과 `shift` 파라미터를 추가했습니다
- 초기 `scale`(1로 곱하기)과 `shift`(0 더하기) 값들은 아무런 효과가 없습니다. 하지만 `scale`과 `shift`는 훈련 가능한 파라미터로, 훈련 과업에서 모델의 성능을 향상시킬 수 있다고 판단되면 LLM이 훈련 중에 자동으로 조정합니다
- 이를 통해 모델이 처리하는 데이터에 가장 적합한 적절한 스케일링과 시프팅을 학습할 수 있습니다
- 또한 분산의 제곱근을 계산하기 전에 작은 값(`eps`)을 추가합니다. 이는 분산이 0인 경우 0으로 나누기 오류를 피하기 위함입니다

**편향된 분산**
- 위의 분산 계산에서 `unbiased=False`를 설정하는 것은 분산을 계산할 때 $\frac{\sum_i (x_i - \bar{x})^2}{n}$ 공식을 사용한다는 의미입니다. 여기서 n은 샘플 크기(여기서는 특성이나 열의 수)입니다. 이 공식은 베셀의 보정(분모에서 `n-1`을 사용)을 포함하지 않아 분산의 편향된 추정치를 제공합니다
- 임베딩 차원 `n`이 매우 큰 LLM의 경우, n과 `n-1` 사용의 차이는 무시할 수 있습니다
- 하지만 GPT-2는 정규화 레이어에서 편향된 분산으로 훈련되었기 때문에, 나중 장에서 로드할 사전 훈련된 가중치와의 호환성을 위해 이 설정을 채택했습니다

- 이제 `LayerNorm`을 실제로 사용해보겠습니다:

In [ ]:
ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)

In [ ]:
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)

print("Mean:\n", mean)
print("Variance:\n", var)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/07.webp" width="400px">

## 4.3 GELU 활성화를 사용한 피드포워드 네트워크 구현하기

- 이 섹션에서는 LLM의 트랜스포머 블록의 일부로 사용되는 작은 신경망 서브모듈을 구현합니다
- 활성화 함수부터 시작합니다
- 딥러닝에서 ReLU(Rectified Linear Unit) 활성화 함수는 다양한 신경망 아키텍처에서 단순성과 효과성으로 인해 일반적으로 사용됩니다
- LLM에서는 전통적인 ReLU 외에도 다양한 다른 유형의 활성화 함수들이 사용됩니다. 두 가지 주목할 만한 예시는 GELU(Gaussian Error Linear Unit)와 SwiGLU(Swish-Gated Linear Unit)입니다
- GELU와 SwiGLU는 각각 가우시안과 시그모이드 게이트 선형 유닛을 통합한 더 복잡하고 부드러운 활성화 함수로, ReLU의 더 간단한 구간선형 함수와 달리 딥러닝 모델에 더 나은 성능을 제공합니다

- GELU([Hendrycks and Gimpel 2016](https://arxiv.org/abs/1606.08415))는 여러 가지 방법으로 구현할 수 있습니다. 정확한 버전은 GELU(x)=x⋅Φ(x)로 정의되며, 여기서 Φ(x)는 표준 가우시안 분포의 누적 분포 함수입니다.
- 실제로는 계산상 더 저렴한 근사를 구현하는 것이 일반적입니다: $\text{GELU}(x) \approx 0.5 \cdot x \cdot \left(1 + \tanh\left[\sqrt{\frac{2}{\pi}} \cdot \left(x + 0.044715 \cdot x^3\right)\right]\right)$ (원래 GPT-2 모델도 이 근사로 훈련되었습니다)

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

In [ ]:
import matplotlib.pyplot as plt

gelu, relu = GELU(), nn.ReLU()

# 샘플 데이터
x = torch.linspace(-3, 3, 100)
y_gelu, y_relu = gelu(x), relu(x)

plt.figure(figsize=(8, 3))
for i, (y, label) in enumerate(zip([y_gelu, y_relu], ["GELU", "ReLU"]), 1):
    plt.subplot(1, 2, i)
    plt.plot(x, y)
    plt.title(f"{label} 활성화 함수")
    plt.xlabel("x")
    plt.ylabel(f"{label}(x)")
    plt.grid(True)

plt.tight_layout()
plt.show()

- 보시다시피 ReLU는 양수이면 입력을 직접 출력하고, 그렇지 않으면 0을 출력하는 구간선형 함수입니다
- GELU는 ReLU를 근사하지만 음수 값에 대해 0이 아닌 기울기를 갖는 부드럽고 비선형인 함수입니다(약 -0.75에서 제외)

- 다음으로, 나중에 LLM의 트랜스포머 블록에서 사용할 작은 신경망 모듈인 `FeedForward`를 구현해보겠습니다:

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
print(GPT_CONFIG_124M["emb_dim"])

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/09.webp?12" width="400px">

In [ ]:
ffn = FeedForward(GPT_CONFIG_124M)

# 입력 형태: [batch_size, num_token, emb_size]
x = torch.rand(2, 3, 768) 
out = ffn(x)
print(out.shape)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/10.webp" width="400px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/11.webp" width="400px">

## 4.4 숏컷 연결 추가하기

- 다음으로, 스킵 또는 잔차 연결이라고도 불리는 숏컷 연결의 개념에 대해 이야기해보겠습니다
- 원래 숏컷 연결은 기울기 소실 문제를 완화하기 위해 컴퓨터 비전을 위한 깊은 네트워크(잔차 네트워크)에서 제안되었습니다
- 숏컷 연결은 기울기가 네트워크를 통해 흐를 수 있는 대안적인 더 짧은 경로를 만듭니다
- 이는 한 레이어의 출력을 나중 레이어의 출력에 더하여 달성되며, 보통 중간의 하나 이상의 레이어를 건너뜁니다
- 작은 예제 네트워크로 이 아이디어를 설명해보겠습니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/12.webp?123" width="400px">

- 코드로는 다음과 같습니다:

In [ ]:
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU())
        ])

    def forward(self, x):
        for layer in self.layers:
            # 현재 레이어의 출력을 계산합니다
            layer_output = layer(x)
            # 숏컷을 적용할 수 있는지 확인합니다
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output
        return x


def print_gradients(model, x):
    # 순전파
    output = model(x)
    target = torch.tensor([[0.]])

    # 타겟과 출력이 얼마나 가까운지에 기반한 손실 계산
    loss = nn.MSELoss()
    loss = loss(output, target)
    
    # 기울기를 계산하기 위한 역전파
    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            # 가중치의 평균 절댓값 기울기를 출력합니다
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

- 먼저 숏컷 연결 **없이** 기울기 값을 출력해보겠습니다:

In [ ]:
layer_sizes = [3, 3, 3, 3, 3, 1]  

sample_input = torch.tensor([[1., 0., -1.]])

torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=False
)
print_gradients(model_without_shortcut, sample_input)

- 다음으로, 숏컷 연결 **있이** 기울기 값을 출력해보겠습니다:

In [ ]:
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=True
)
print_gradients(model_with_shortcut, sample_input)

- 위의 출력에서 알 수 있듯이, 숏컷 연결은 초기 레이어(`layer.0` 방향)에서 기울기가 소실되는 것을 방지합니다
- 다음에 트랜스포머 블록을 구현할 때 이 숏컷 연결 개념을 사용할 것입니다

## 4.5 트랜스포머 블록에서 어텐션과 선형 레이어 연결하기

- 이 섹션에서는 이제 이전 개념들을 소위 트랜스포머 블록으로 결합합니다
- 트랜스포머 블록은 이전 장의 인과 멀티헤드 어텐션 모듈을 앞서 섹션에서 구현한 선형 레이어인 피드포워드 신경망과 결합합니다
- 또한 트랜스포머 블록은 드롭아웃과 숏컷 연결도 사용합니다

In [ ]:
# `previous_chapters.py` 파일이 로컬에서 사용할 수 없는 경우,
# `llms-from-scratch` PyPI 패키지에서 가져올 수 있습니다.
# 자세한 내용은 https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg 를 참조하세요
# 예를 들어:
# from llms_from_scratch.ch03 import MultiHeadAttention

from previous_chapters import MultiHeadAttention


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # 어텐션 블록에 대한 숏컷 연결
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # 형태 [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력을 다시 더합니다

        # 피드포워드 블록에 대한 숏컷 연결
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력을 다시 더합니다

        return x

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/13.webp?1" width="400px">

- 각각 6개의 토큰을 가진 2개의 입력 샘플이 있다고 가정해보겠습니다. 여기서 각 토큰은 768차원 임베딩 벡터입니다. 그러면 이 트랜스포머 블록은 셀프 어텐션을 적용한 다음 선형 레이어를 적용하여 비슷한 크기의 출력을 생성합니다
- 출력을 이전 장에서 논의한 컨텍스트 벡터의 증강된 버전으로 생각할 수 있습니다

In [ ]:
torch.manual_seed(123)

x = torch.rand(2, 4, 768)  # 형태: [batch_size, num_tokens, emb_dim]
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/14.webp?1" width="400px">

## 4.6 GPT 모델 코딩하기

- 거의 완료되었습니다: 이제 트랜스포머 블록을 이 장의 맨 처음에 코딩한 아키텍처에 연결하여 사용 가능한 GPT 아키텍처를 얻어보겠습니다
- 트랜스포머 블록이 여러 번 반복된다는 점에 주목하세요. 가장 작은 1억 2400만 GPT-2 모델의 경우, 12번 반복합니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/15.webp" width="400px">

- `cfg["n_layers"] = 12`인 해당 코드 구현:

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # 형태 [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

- 1억 2400만 파라미터 모델의 구성을 사용하여, 이제 랜덤 초기 가중치로 이 GPT 모델을 인스턴스화할 수 있습니다:

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

- 다음 장에서 이 모델을 훈련할 것입니다
- 하지만 크기에 대한 빠른 참고사항: 우리는 이전에 이를 1억 2400만 파라미터 모델이라고 언급했습니다. 이 숫자를 다음과 같이 확인할 수 있습니다:

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

- 위에서 보듯이, 이 모델은 1억 2400만이 아닌 1억 6300만 파라미터를 가지고 있습니다. 왜 그럴까요?
- 원래 GPT-2 논문에서 연구자들은 가중치 타이잉을 적용했습니다. 이는 토큰 임베딩 레이어(`tok_emb`)를 출력 레이어로 재사용한다는 의미로, `self.out_head.weight = self.tok_emb.weight`를 설정하는 것입니다
- 토큰 임베딩 레이어는 50,257차원의 원-핫 인코딩된 입력 토큰을 768차원 임베딩 표현으로 투영합니다
- 출력 레이어는 768차원 임베딩을 다시 50,257차원 표현으로 투영하여 이를 다시 단어로 변환할 수 있게 합니다(다음 섹션에서 더 자세히 다룰 예정)
- 따라서 임베딩과 출력 레이어는 가중치 행렬의 형태에서 볼 수 있듯이 동일한 수의 가중치 파라미터를 가집니다

In [ ]:
print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)

- 원래 GPT-2 논문에서 연구자들은 토큰 임베딩 행렬을 출력 행렬로 재사용했습니다
- 이에 따라 출력 레이어의 파라미터 수를 뺀다면 1억 2400만 파라미터 모델이 됩니다:

In [ ]:
total_params_gpt2 =  total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")

- 실제로는 가중치 타이잉 없이 모델을 훈련하는 것이 더 쉽다는 것을 발견했기 때문에 여기서는 구현하지 않았습니다
- 하지만 5장에서 사전 훈련된 가중치를 로드할 때 이 가중치 타이잉 아이디어를 다시 방문하고 적용할 것입니다
- 마지막으로, 유용한 참조점이 될 수 있는 모델의 메모리 요구사항을 다음과 같이 계산할 수 있습니다:

In [ ]:
# 총 크기를 바이트로 계산 (float32로 가정, 파라미터당 4바이트)
total_size_bytes = total_params * 4

# 메가바이트로 변환
total_size_mb = total_size_bytes / (1024 * 1024)

print(f"Total size of the model: {total_size_mb:.2f} MB")

- 연습: [GPT-2 논문](https://scholar.google.com/citations?view_op=view_citation&hl=en&user=dOad5HoAAAAJ&citation_for_view=dOad5HoAAAAJ:YsMSGLbcyi4C)에서 참조되는 다음과 같은 다른 구성들도 시도해볼 수 있습니다.

    - **GPT2-small** (우리가 이미 구현한 1억 2400만 구성):
        - "emb_dim" = 768
        - "n_layers" = 12
        - "n_heads" = 12

    - **GPT2-medium:**
        - "emb_dim" = 1024
        - "n_layers" = 24
        - "n_heads" = 16
    
    - **GPT2-large:**
        - "emb_dim" = 1280
        - "n_layers" = 36
        - "n_heads" = 20
    
    - **GPT2-XL:**
        - "emb_dim" = 1600
        - "n_layers" = 48
        - "n_heads" = 25

## 4.7 텍스트 생성하기

- 위에서 구현한 GPT 모델과 같은 LLM은 한 번에 한 단어씩 생성하는 데 사용됩니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/16.webp" width="400px">

- 다음 `generate_text_simple` 함수는 텍스트를 생성하는 간단하고 빠른 방법인 그리디 디코딩을 구현합니다
- 그리디 디코딩에서는 각 단계에서 모델이 다음 출력으로 가장 높은 확률을 가진 단어(또는 토큰)를 선택합니다(가장 높은 로짓이 가장 높은 확률에 해당하므로 실제로는 소프트맥스 함수를 명시적으로 계산할 필요도 없습니다)
- 다음 장에서는 더 고급 `generate_text` 함수를 구현할 것입니다
- 아래 그림은 GPT 모델이 입력 컨텍스트가 주어졌을 때 다음 단어 토큰을 어떻게 생성하는지를 보여줍니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/17.webp" width="600px">

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx는 현재 컨텍스트의 인덱스 배열 (batch, n_tokens)
    for _ in range(max_new_tokens):
        
        # 현재 컨텍스트가 지원되는 컨텍스트 크기를 초과하면 자르기
        # 예: LLM이 5개 토큰만 지원하고 컨텍스트 크기가 10이면
        # 마지막 5개 토큰만 컨텍스트로 사용됩니다
        idx_cond = idx[:, -context_size:]
        
        # 예측을 얻습니다
        with torch.no_grad():
            logits = model(idx_cond)
        
        # 마지막 시간 단계에만 집중합니다
        # (batch, n_tokens, vocab_size)가 (batch, vocab_size)가 됩니다
        logits = logits[:, -1, :]  

        # 소프트맥스를 적용하여 확률을 얻습니다
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # 가장 높은 확률값을 가진 어휘 항목의 idx를 얻습니다
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # 실행 중인 시퀀스에 샘플링된 인덱스를 추가합니다
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

- 위의 `generate_text_simple`은 한 번에 하나의 토큰을 생성하는 반복적인 프로세스를 구현합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/18.webp" width="600px">

- 입력 예제를 준비해보겠습니다:

In [ ]:
start_context = "Hello, I am"

encoded = tokenizer.encode(start_context)
print("encoded:", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

In [ ]:
model.eval() # 드롭아웃 비활성화

out = generate_text_simple(
    model=model,
    idx=encoded_tensor, 
    max_new_tokens=6, 
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output:", out)
print("Output length:", len(out[0]))

- 배치 차원을 제거하고 텍스트로 다시 변환합니다:

In [ ]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

- 모델이 훈련되지 않았기 때문에 위의 랜덤한 출력 텍스트가 나온다는 점에 유의하세요
- 다음 장에서 모델을 훈련할 것입니다

## 요약 및 핵심 사항

- 이 Jupyter 노트북에서 구현한 GPT 모델이 포함된 독립형 스크립트인 [./gpt.py](./gpt.py) 스크립트를 참조하세요
- 연습 문제 해답은 [./exercise-solutions.ipynb](./exercise-solutions.ipynb)에서 찾을 수 있습니다